# Notebook 2: Feature Analysis

Deep-dive into the 18-dimensional feature vector:
- Correlation heatmap with target MCS
- Distribution of each feature by MCS class
- Feature importance via mutual information
- PCA / t-SNE visualisation of the feature space

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from src.channel.simulator import ChannelSimulator, ChannelConfig

sns.set_theme(style='whitegrid', font_scale=1.0)
print('Imports OK')

## 1. Generate Dataset

In [ ]:
cfg = ChannelConfig(channel_type='rayleigh', snr_mean_db=15.0, snr_std_db=6.0, seed=42)
sim = ChannelSimulator(cfg)
ds  = sim.generate_dataset(n_frames=50_000, window=8)

print(ds.summary())

X = ds.features
y = ds.labels_mcs
feat_names = ds.feature_names

df = pd.DataFrame(X, columns=feat_names)
df['mcs'] = y
print(f'\nDataFrame shape: {df.shape}')
df.describe().T

## 2. Correlation Matrix

In [ ]:
# Drop one-hot channel features for clarity
numeric_cols = [c for c in feat_names if not c.startswith('ch_')] + ['mcs']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1,
            annot=True, fmt='.2f', linewidths=0.4,
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Feature Correlation Matrix (Rayleigh Fading)', pad=12)
plt.tight_layout()
plt.savefig('../results/plots/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with MCS
print('\nTop correlations with MCS target:')
print(corr['mcs'].drop('mcs').abs().sort_values(ascending=False).head(10))

## 3. Mutual Information with Target MCS

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

mi_scores = mutual_info_classif(X_scaled, y, n_neighbors=5, random_state=42)
mi_df = pd.DataFrame({'feature': feat_names, 'MI': mi_scores}).sort_values('MI', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#4C72B0' if not f.startswith('ch_') else '#8172B2' for f in mi_df['feature']]
ax.barh(mi_df['feature'], mi_df['MI'], color=colors, alpha=0.85, edgecolor='white')
ax.set_xlabel('Mutual Information with MCS Target')
ax.set_title('Feature Importance: Mutual Information')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most informative features:')
print(mi_df.tail(5).to_string(index=False))

## 4. SNR Window History vs MCS — Box Plots

In [ ]:
# Group MCS into 4 broad categories for readability
def mcs_group(m):
    if m <= 4: return 'Low (0–4)\nQPSK'
    elif m <= 8: return 'Mid-Low (5–8)\n16-QAM'
    elif m <= 13: return 'Mid-High (9–13)\n64-QAM'
    else: return 'High (14–18)\n256-QAM'

df['mcs_group'] = df['mcs'].apply(mcs_group)
group_order = ['Low (0–4)\nQPSK', 'Mid-Low (5–8)\n16-QAM',
               'Mid-High (9–13)\n64-QAM', 'High (14–18)\n256-QAM']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
key_features = ['snr_t', 'cqi_t', 'snr_mean_w', 'snr_std_w', 'snr_diff1', 'snr_diff2']

for ax, feat in zip(axes.flat, key_features):
    # Sample for speed
    sample = df.sample(5000, random_state=42)
    sns.boxplot(data=sample, x='mcs_group', y=feat,
                order=group_order, palette='Set2', ax=ax,
                width=0.5, linewidth=1)
    ax.set_xlabel('')
    ax.set_ylabel(feat)
    ax.set_title(f'Distribution of {feat}')
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Feature Distributions by MCS Group (Rayleigh Fading)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/feature_by_mcs_group.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. PCA: Feature Space Visualisation

In [ ]:
pca = PCA(n_components=2, random_state=42)
idx = np.random.choice(len(X), 8000, replace=False)
X_pca = pca.fit_transform(X_scaled[idx])
y_sample = y[idx]

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=y_sample, cmap='viridis', s=4, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Optimal MCS Index')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of Feature Space (coloured by Optimal MCS)')

print(f'Total variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')
print('\nPC1 loadings (top 5):')
loadings = pd.Series(pca.components_[0], index=feat_names).abs().sort_values(ascending=False)
print(loadings.head(5))

plt.tight_layout()
plt.savefig('../results/plots/pca_feature_space.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key findings from feature analysis:

- **`snr_t` and `cqi_t`** have the highest mutual information with MCS target — confirming SNR level is the primary driver
- **`snr_diff1`** (1st temporal difference) is the 3rd most important feature — channel dynamics matter
- **`snr_std_w`** differentiates slow-fading (low std) from fast-fading (high std) environments
- PCA shows that the first two principal components explain ~75% of variance, and the MCS labels form a clear gradient in this 2D space — confirming learnability
- Channel type OHE features have low mutual information individually but are important as interaction modifiers